# Dev experiments: OSTrack + V-JEPA (Kaggle, self-contained)

Everything in one session, no notebook/dataset inputs (mounting large outputs fails on Kaggle).
Choose the steps in `RUN` (cell 4). The V-JEPA cache lives only in this session, so every
V-JEPA step needs `cache=True`.

| step | what | time |
|---|---|---|
| 0 | self-test on 30 frames (fails within minutes if something is broken) | ~5 min |
| 1 | V-JEPA cache for the 40 dev videos | ~1 h |
| 2 | OSTrack + V-JEPA re-localization, no verification | ~30 min |
| 2b | re-localization verified by OSTrack (V-JEPA proposes 3 places) | ~30 min |
| 3 | training samples from 280 train videos | ~1 h |
| 4 | train fusion_v1 | ~30-60 min |
| 5 | OSTrack + fusion_v1 on dev | ~30 min |

Results are packed into `/kaggle/working/dev_experiments.zip` after every step.

Settings: **Accelerator = GPU T4 x2**, **Internet = On**, no inputs needed. Then **Save Version -> Save & Run All**.

In [ ]:
# fail fast with a clear message if the session has no internet or no GPU
import socket
try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError:
    raise SystemExit('NO INTERNET: Settings (right panel) -> Internet = On '
                     '(needs a phone-verified Kaggle account), then run again.')
import torch
if torch.cuda.device_count() == 0:
    raise SystemExit('NO GPU: Settings (right panel) -> Accelerator = GPU T4 x2, then run again.')
print(torch.cuda.device_count(), 'GPUs:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess  # check=True stops the run here if cloning fails
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/MariiaHamanuk/Object-Tracking.git',
                '/kaggle/working/Object-Tracking'], check=True)
!pip install -q timm==0.6.13 easydict visdom tensorboardX jpeg4py lmdb gdown "transformers>=4.53"
!gdown 1dySfPrSk-knAEo0lTXqZ1kMk3BCvVofV -O /kaggle/working/OSTrack_ep0300.pth.tar

In [ ]:
# which steps to run
RUN = dict(cache=True,
           reloc=False,          # done: results_raw/ostrack_vjepa_reloc (AUC 55.58 on dev)
           reloc_verify=True,    # V-JEPA proposes 3 places, OSTrack verifies them on the same frame
           prepare=False, train=False, fusion=False)   # fusion_v1 done: results_raw/ostrack_fusion_v1

In [ ]:
import os, glob, subprocess, sys, torch, numpy as np
os.chdir('/kaggle/working/Object-Tracking')
sys.path.insert(0, '.')
os.makedirs('/kaggle/tmp', exist_ok=True)
W = '/kaggle/working'
CKPT = f'{W}/OSTrack_ep0300.pth.tar'
CACHE = '/kaggle/tmp/cache/vjepa'
DEV = 'configs/splits/dev_occlusion.txt'
N_GPU = max(torch.cuda.device_count(), 1)
os.makedirs(f'{W}/logs', exist_ok=True)

def run_shards(name, args):
    """Run a script once per GPU (each takes every N-th video), wait, show the log tail."""
    procs = []
    for i in range(N_GPU):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
        log = open(f'{W}/logs/{name}_shard{i}.log', 'w')
        procs.append(subprocess.Popen(['python'] + args + ['--shard', str(i), '--num-shards', str(N_GPU)],
                                      env=env, stdout=log, stderr=subprocess.STDOUT))
    codes = [p.wait() for p in procs]
    os.system(f"grep -hE 'FAILED|Error|Traceback' {W}/logs/{name}_shard*.log | head -n 20")
    os.system(f"tail -n 2 {W}/logs/{name}_shard*.log")
    print(name, 'exit codes', codes)
    return codes

def pack():
    """Zip everything worth keeping (small files only) - called after every step."""
    os.system(f'cd {W} && rm -f dev_experiments.zip && zip -qr dev_experiments.zip results_raw logs output 2>/dev/null; ls -la {W}/dev_experiments.zip')

## 0. Self-test (30 frames)
Plain OSTrack, re-localization and zero-initialised fusion must run; fusion before training must equal
plain OSTrack; a wrong candidate offered for verification must be rejected and leave the track unchanged.

In [ ]:
from src.tracker.fusion import FusionV1
os.makedirs('/kaggle/tmp/selftest/cache', exist_ok=True)
np.savez('/kaggle/tmp/selftest/cache/airplane-1.npz', frames=np.arange(0, 3000, 8),
         maps=np.random.randn(375, 16, 16, 1024).astype(np.float16))
torch.save(FusionV1().state_dict(), '/kaggle/tmp/selftest/fusion_zero.pth')
base = ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', 'airplane-1', '--max-frames', '30',
        '--work', '/kaggle/tmp/selftest/work']
for name, extra in [('plain', []),
                    ('reloc', ['--vjepa-cache', '/kaggle/tmp/selftest/cache', '--reloc-verify', '3']),
                    ('fusion0', ['--vjepa-cache', '/kaggle/tmp/selftest/cache', '--fusion', '/kaggle/tmp/selftest/fusion_zero.pth'])]:
    subprocess.run(['python'] + base + ['--out', f'/kaggle/tmp/selftest/{name}'] + extra, check=True)
plain = np.loadtxt('/kaggle/tmp/selftest/plain/airplane-1.txt', delimiter='\t')
fusion0 = np.loadtxt('/kaggle/tmp/selftest/fusion0/airplane-1.txt', delimiter='\t')
print('max box difference, zero-init fusion vs plain OSTrack:', np.abs(plain - fusion0).max())
assert np.abs(plain - fusion0).max() < 1.0, 'zero-initialised fusion should reproduce OSTrack'

from src.tracker.ostrack import OSTrackRunner
from src.data.remote_zip import open_class_zip, extract_frames
fdir = extract_frames(open_class_zip('airplane'), 'airplane-1', range(30), '/kaggle/tmp/selftest/frames')
frames = sorted((fdir / 'img').glob('*.jpg'))[:30]
gt = np.loadtxt(fdir / 'groundtruth.txt', delimiter=',')
runner = OSTrackRunner(CKPT, save_dir='/kaggle/tmp/selftest/ost')
corner = lambda i, b, s: np.array([[0, 0, b[2], b[3]]]) if i % 10 == 5 else None   # a wrong place
res = runner.track(frames, gt[0], hook=corner, verify=True)
print('verification checks:', res['checks'])
assert len(res['checks']) == 3 and not any(c['accepted'] for c in res['checks']), 'wrong candidate was accepted'
print('max box difference, verified run vs plain OSTrack:', np.abs(res['boxes'] - plain).max())
assert np.abs(res['boxes'] - plain).max() < 1.0, 'rejected candidates must leave the track unchanged'
del runner
torch.cuda.empty_cache()
print('SELF-TEST OK')

## 1. V-JEPA cache for the dev videos

In [ ]:
if RUN['cache']:
    for attempt in range(2):  # a re-run only does the videos that failed to download
        run_shards('vjepa_cache', ['scripts/cache_vjepa.py', '--sequences', DEV, '--out', CACHE, '--work', '/kaggle/tmp'])
        n = len(glob.glob(f'{CACHE}/*.npz'))
        print(n, 'of 40 dev videos cached')
        if n == 40:
            break
    assert n == 40, 'V-JEPA cache incomplete'

## 2. OSTrack + V-JEPA re-localization on dev (no verification)

In [ ]:
if RUN['reloc']:
    run_shards('reloc', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV, '--vjepa-cache', CACHE,
                         '--out', f'{W}/results_raw/ostrack_vjepa_reloc/lasot', '--work', '/kaggle/tmp'])
    pack()

## 2b. Re-localization verified by OSTrack (V-JEPA proposes 3 places, OSTrack decides)

In [ ]:
if RUN['reloc_verify']:
    run_shards('reloc_verify', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV,
                                '--vjepa-cache', CACHE, '--reloc-verify', '3',
                                '--out', f'{W}/results_raw/ostrack_vjepa_reloc_verify/lasot', '--work', '/kaggle/tmp'])
    pack()

## 3. Training samples (280 train videos, 4 per class)

In [ ]:
if RUN['prepare']:
    run_shards('prepare', ['scripts/prepare_fusion_data.py', '--out', '/kaggle/tmp/fusion_data', '--work', '/kaggle/tmp'])
    print(len(glob.glob('/kaggle/tmp/fusion_data/motion/*.npz')), 'videos with samples')

## 4. Train fusion_v1 (one GPU)

In [ ]:
if RUN['train']:
    subprocess.run(['python', 'scripts/train_fusion.py', '--data', '/kaggle/tmp/fusion_data/motion', '--checkpoint', CKPT,
                    '--out', f'{W}/output/fusion_v1'], check=True)
    pack()

## 5. OSTrack + fusion_v1 on dev

In [ ]:
if RUN['fusion']:
    run_shards('fusion', ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', DEV, '--vjepa-cache', CACHE,
                          '--fusion', f'{W}/output/fusion_v1/fusion.pth',
                          '--out', f'{W}/results_raw/ostrack_fusion_v1/lasot', '--work', '/kaggle/tmp'])
    pack()

In [ ]:
pack()
os.system(f'rm -rf {W}/Object-Tracking {CKPT}; du -sh {W}/*')